# Optimizing $\chi$ Through the Simulation

Start from a random 16-component $\chi$ and optimize a soft phase-count loss directly through the IMEX simulation.

In [ ]:
# Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import sys
# sys.path.insert(0, '/content/drive/MyDrive/phase_separation')
#
# !cp -r "/content/drive/MyDrive/phase_separation" /content/phase_separation
# %cd phase_separation
# !ls
# !pip install -r requirements.txt
# !pip install jax-tqdm scikit-learn

import os
import sys

sys.path.insert(0, os.path.abspath('../..'))
# !pip install optax

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from jax_phase_separation.solver import (
    SimulationParams, simulate, simulate_with_snapshots,
    _make_step_fn, make_wavenumbers,
)
from jax_phase_separation.free_energy import compute_jacobian
from jax_phase_separation.utils import (
    generate_initial_conditions, build_params,
    plot_volume_fractions, plot_phase_map, plot_partition_ratios,
)
from jax_phase_separation.analysis import analyse_snapshot
from jax_tqdm import scan_tqdm

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## 1. Setup

$\chi$ is symmetric with zero diagonal, so we parametrize it by its
$N(N{-}1)/2$ upper-triangular elements.

In [ ]:
N_COM = 16
N_GRID = 64
BETA = N_COM / (N_COM + 1.0)
DT = 5e-6
LMBDA = 0.01
TARGET_PHASES = 6

n_upper = N_COM * (N_COM - 1) // 2
print(f'Components: {N_COM}, free parameters: {n_upper}')
print(f'Target: {TARGET_PHASES} phases')

In [ ]:
triu_rows, triu_cols = jnp.triu_indices(N_COM, k=1)

def chi_from_upper(upper_tri):
    """Reconstruct symmetric zero-diagonal chi from upper-triangle vector."""
    chi = jnp.zeros((N_COM, N_COM))
    chi = chi.at[triu_rows, triu_cols].set(upper_tri)
    return chi + chi.T

def make_params(chi):
    """Build SimulationParams from chi (JAX-traceable, for use inside grad)."""
    kappa = jnp.eye(N_COM) * 1.0
    chi_s = jnp.zeros((N_COM, N_COM))
    r = jnp.ones((N_COM, 1))
    A = jnp.max(jnp.abs(chi)) * LMBDA
    return SimulationParams(
        chi=chi, kappa=kappa, chi_s=chi_s, r=r,
        lmbda=LMBDA, dt=DT, kon=0.0, koff=0.0, A=A,
    )

### Initial random $\chi$

In [ ]:
key = jax.random.PRNGKey(7)
k_chi, k_ic = jax.random.split(key)

SIGMA_INIT = 4.8
raw = SIGMA_INIT * jax.random.normal(k_chi, (N_COM, N_COM))
chi_init_full = (raw + raw.T) / 2.0
chi_init_full = chi_init_full.at[jnp.diag_indices(N_COM)].set(0.0)
upper_init = chi_init_full[triu_rows, triu_cols]

chi_s_zero = jnp.zeros(N_COM)
r_ones = jnp.ones(N_COM)

J_init = compute_jacobian(N_COM, BETA, chi_init_full, chi_s_zero, r_ones)
w_init = jnp.linalg.eigvalsh(J_init)
n_neg_init = int(jnp.sum(w_init < 0))

print(f'Initial unstable modes: {n_neg_init}')
print(f'Eigenvalues: {np.array(w_init).round(2)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

im = axes[0].imshow(chi_init_full, cmap='RdBu_r', origin='lower')
plt.colorbar(im, ax=axes[0], label=r'$\chi_{ij}$')
axes[0].set_title('Initial $\\chi$ (random)')

colors = ['tab:red' if e < 0 else 'tab:blue' for e in w_init]
axes[1].bar(range(N_COM), w_init, color=colors)
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('mode')
axes[1].set_ylabel('eigenvalue')
axes[1].set_title(f'Hessian spectrum ({n_neg_init} unstable)')
fig.tight_layout()
plt.show()

## 2. Differentiable loss through the simulation

The loss compares a soft PCA phase count of the final field with the target, plus optional $L_2$ and spectral regularization. The simulation scan is checkpointed in chunks to keep backpropagation memory bounded.

In [ ]:
N_GRID_OPT = 32
N_STEPS_OPT = 200_000
CHUNK_SIZE = 500
N_CHUNKS = N_STEPS_OPT // CHUNK_SIZE
PHASE_THRESH = 9e-3
SIGMOID_TEMP = 1000.0

c0_small = generate_initial_conditions(
    N_COM, N_GRID_OPT, beta=BETA, noise_strength=0.01, key=k_ic,
)

def soft_phase_count(c):
    """Differentiable proxy for number of coexisting phases."""
    c_sol = jnp.maximum(1.0 - jnp.sum(c, axis=0, keepdims=True), 1e-30)
    c_all = jnp.concatenate([c, c_sol], axis=0)
    N_ch = c_all.shape[0]
    flat = c_all.reshape(N_ch, -1).T

    mean = flat.mean(axis=0)
    std = jnp.maximum(flat.std(axis=0), 1e-12)
    centered = (flat - mean) / std

    cov = centered.T @ centered / (centered.shape[0] - 1)
    eigvals = jnp.linalg.eigvalsh(cov)[::-1]
    return jnp.sum(jax.nn.sigmoid(SIGMOID_TEMP * (eigvals - PHASE_THRESH)))


SPECTRAL_WEIGHT = 0.1   # set to 0 to disable spectral regularisation
MARGIN = 0.5

def spectral_reg(upper_tri):
    """Penalise having fewer than TARGET_PHASES unstable Hessian modes.

    Uses softplus barriers around the eigenvalue sign boundaries so that
    the gradient smoothly pushes eigenvalues into the desired regime.
    """
    chi = chi_from_upper(upper_tri)
    J = compute_jacobian(N_COM, BETA, chi, chi_s_zero, r_ones)
    w = jnp.linalg.eigvalsh(J)
    target_neg = TARGET_PHASES - 1
    loss_gap = (
        jax.nn.softplus(w[target_neg - 1] + MARGIN)
        + jax.nn.softplus(-(w[target_neg] - MARGIN))
    )
    return loss_gap


def sim_loss(upper_tri):
    chi = chi_from_upper(upper_tri)
    params = make_params(chi)

    wn = make_wavenumbers(N_GRID_OPT)
    Ainv = 1.0 / (1.0 + params.A * wn["k4"] * params.dt)
    cHat0 = jnp.fft.fftn(c0_small, axes=(-2, -1))
    inner_step = _make_step_fn(params, wn, Ainv, False)

    @scan_tqdm(N_CHUNKS, print_rate=max(1, N_CHUNKS // 20))
    @jax.checkpoint
    def chunk(carry, x):
        carry, _ = jax.lax.scan(inner_step, carry, None, length=CHUNK_SIZE)
        return carry, None

    (c_final, _), _ = jax.lax.scan(
        chunk, (c0_small, cHat0), jnp.arange(N_CHUNKS), length=N_CHUNKS,
    )

    n_soft = soft_phase_count(c_final)
    l2 = 1e-4 * jnp.sum(upper_tri ** 2)
    spec = SPECTRAL_WEIGHT * spectral_reg(upper_tri)
    return (n_soft - TARGET_PHASES) ** 2 + l2 + spec

sim_value_and_grad = jax.value_and_grad(sim_loss)

In [ ]:
import optax

LR_INIT = 0.05
schedule = optax.cosine_decay_schedule(init_value=LR_INIT, decay_steps=20, alpha=0.1)
optimizer = optax.adam(learning_rate=schedule)

### JIT warm-up

The first call compiles both the forward and backward passes.
The progress bar covers the forward pass; the backward pass
(gradient computation) takes a similar amount of time afterwards.

In [ ]:
%%time
loss_v, grad_v = sim_value_and_grad(upper_init)
jax.block_until_ready(grad_v)
print(f'Initial sim loss: {float(loss_v):.4f}')
print(f'Gradient norm:    {float(jnp.linalg.norm(grad_v)):.4e}')

## 3. Optimization loop

In [ ]:
%%time
N_ITERS = 30
upper_tri = upper_init.copy()
opt_state = optimizer.init(upper_tri)
loss_history = []
best_loss = float('inf')
best_upper_tri = upper_tri.copy()

for i in range(N_ITERS):
    loss_val, grad = sim_value_and_grad(upper_tri)
    jax.block_until_ready(grad)
    updates, opt_state = optimizer.update(grad, opt_state)
    upper_tri = optax.apply_updates(upper_tri, updates)

    lr_now = float(schedule(opt_state[1].count - 1))
    lv = float(loss_val)
    loss_history.append(lv)
    if lv < best_loss:
        best_loss = lv
        best_upper_tri = upper_tri.copy()
        tag = ' *best*'
    else:
        tag = ''
    print(f'iter {i:2d}  lr={lr_now:.5f}  loss={lv:.4f}{tag}')

upper_tri = best_upper_tri
print(f'\nOptimization complete.  Best loss: {best_loss:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_history, 'o-')
ax.set_xlabel('iteration')
ax.set_ylabel('loss')
ax.set_title('Simulation-based optimization')
ax.set_yscale('log')
fig.tight_layout()
plt.show()

## 4. What changed?

Compare the Hessian eigenvalue spectrum before and after optimization.

In [ ]:
chi_opt = chi_from_upper(upper_tri)
J_opt = compute_jacobian(N_COM, BETA, chi_opt, chi_s_zero, r_ones)
w_opt = jnp.linalg.eigvalsh(J_opt)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

im = axes[0].imshow(chi_opt, cmap='RdBu_r', origin='lower')
plt.colorbar(im, ax=axes[0], label=r'$\chi_{ij}$')
axes[0].set_title('Optimized $\\chi$')

for ax, w, title in zip(axes[1:], [w_init, w_opt],
                         ['Random $\\chi$', 'Optimized $\\chi$']):
    colors = ['tab:red' if e < 0 else 'tab:blue' for e in w]
    ax.bar(range(N_COM), w, color=colors)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel('mode')
    ax.set_ylabel('eigenvalue')
    n = int(jnp.sum(jnp.array(w) < 0))
    ax.set_title(f'{title}  ({n} unstable)')

fig.tight_layout()
plt.show()

## 5. Full-resolution verification

Run a long simulation at the full 64×64 grid with both the random
and optimized $\chi$ and compare the resulting phase structure.

In [ ]:
c0 = generate_initial_conditions(N_COM, N_GRID, beta=BETA, noise_strength=0.01, key=k_ic)
N_STEPS_VERIFY = 500_000

In [ ]:
%%time
params_opt = make_params(chi_opt)
c_final_opt = simulate(c0, params_opt, N_GRID, N_STEPS_VERIFY, progress_bar=True)
jax.block_until_ready(c_final_opt)
result_opt = analyse_snapshot(np.array(c_final_opt))
print(f"Optimized chi  → {result_opt['n_phases']} phases  (target: {TARGET_PHASES})")

In [ ]:
%%time
params_init = build_params(chi_init_full, N_COM, beta=BETA, lmbda=LMBDA, dt=DT)
c_final_init = simulate(c0, params_init, N_GRID, N_STEPS_VERIFY, progress_bar=True)
jax.block_until_ready(c_final_init)
result_init = analyse_snapshot(np.array(c_final_init))
print(f"Random chi     → {result_init['n_phases']} phases")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# --- Top row: random chi ---
im0 = axes[0, 0].imshow(chi_init_full, cmap='RdBu_r', origin='lower')
plt.colorbar(im0, ax=axes[0, 0])
axes[0, 0].set_title('Random $\\chi$')

plot_phase_map(result_init['labels'], result_init['n_phases'], ax=axes[0, 1])
axes[0, 1].set_title(f"Phase map — {result_init['n_phases']} phases")

plot_partition_ratios(result_init['partitions'], ax=axes[0, 2])
axes[0, 2].set_title('Partition ratios')

# --- Bottom row: optimized chi ---
im1 = axes[1, 0].imshow(chi_opt, cmap='RdBu_r', origin='lower')
plt.colorbar(im1, ax=axes[1, 0])
axes[1, 0].set_title('Optimized $\\chi$')

plot_phase_map(result_opt['labels'], result_opt['n_phases'], ax=axes[1, 1])
axes[1, 1].set_title(f"Phase map — {result_opt['n_phases']} phases")

plot_partition_ratios(result_opt['partitions'], ax=axes[1, 2])
axes[1, 2].set_title('Partition ratios')

fig.suptitle(f'Random $\\chi$ vs. simulation-optimized $\\chi$ (target: {TARGET_PHASES} phases)',
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
fig, _ = plot_volume_fractions(c_final_opt, ncols=4, vmax=0.75)
fig.suptitle('Optimized $\\chi$ — steady-state volume fractions', y=1.02, fontsize=13)
plt.show()